# **Group Health Analytics Using PySpark**

# Part A - Dataset Setup & Loading

### Connect Google Drive in Colab

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Install Required Libraries

### **Install PySpark**

In [2]:
!pip install pyspark

### **Create a Spark session**

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Health Data Analysis") \
    .getOrCreate()

### **Spark SQL Functions (for calculations & conditions)**

In [4]:
from pyspark.sql.functions import (
    col, when, avg, sum, count, desc
)

### **Spark Data Types**

In [5]:
from pyspark.sql.types import *

### Load the dataset

In [6]:
file_path = "/content/drive/MyDrive/BigData /biodata_advanced_100 (1).csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)

### Display First 5 Rows

In [7]:
df.show(5)

+----------+--------+---+------+---------+---------+-----------+----------+----+-----------+------------+-----------------+
|student_id|    name|age|gender|height_cm|weight_kg|blood_group|department|year|systolic_bp|diastolic_bp|cholesterol_mg_dl|
+----------+--------+---+------+---------+---------+-----------+----------+----+-----------+------------+-----------------+
|      S001|  Minuri| 24|     F|        0|       84|         A-|        IT|   3|        114|          92|                0|
|      S002|  Methma| 21|     F|        0|       58|         B+|        IT|   4|        136|          86|              287|
|      S003|Thivanga| 22|     M|      188|        0|         O-|        IT|   2|        111|          81|              274|
|      S004|Maleesha| 22|     F|      171|        0|         O-|        IT|   2|        110|         102|              238|
|      S005| Randeep| 25|     M|        0|        0|         O+|        DS|   3|        150|          82|                0|
+-------

### Dataset Shape

In [8]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 100
Columns: 12


### Data Schema

In [9]:
df.printSchema()

root
 |-- student_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- height_cm: integer (nullable = true)
 |-- weight_kg: integer (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- department: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- systolic_bp: integer (nullable = true)
 |-- diastolic_bp: integer (nullable = true)
 |-- cholesterol_mg_dl: integer (nullable = true)



# Part B – Data Cleaning & Preprocessing

### Identify Issues

In [10]:
df.describe().show()

+-------+----------+------+------------------+------+-----------------+----------------+-----------+----------+------------------+------------------+------------------+------------------+
|summary|student_id|  name|               age|gender|        height_cm|       weight_kg|blood_group|department|              year|       systolic_bp|      diastolic_bp| cholesterol_mg_dl|
+-------+----------+------+------------------+------+-----------------+----------------+-----------+----------+------------------+------------------+------------------+------------------+
|  count|       100|   100|               100|   100|              100|             100|        100|       100|               100|               100|               100|               100|
|   mean|      NULL|  NULL|             22.86|  NULL|            96.05|           39.33|       NULL|      NULL|              2.93|            136.21|             86.14|            113.96|
| stddev|      NULL|  NULL|1.4565768643007349|  NULL|86.0311

In [11]:
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+----------+----+---+------+---------+---------+-----------+----------+----+-----------+------------+-----------------+
|student_id|name|age|gender|height_cm|weight_kg|blood_group|department|year|systolic_bp|diastolic_bp|cholesterol_mg_dl|
+----------+----+---+------+---------+---------+-----------+----------+----+-----------+------------+-----------------+
|         0|   0|  0|     0|        0|        0|          0|         0|   0|          0|           0|                0|
+----------+----+---+------+---------+---------+-----------+----------+----+-----------+------------+-----------------+



### Replace Invalid Values (0 → NULL)

In [ ]:
df = df.withColumn(
    "height_cm",
    when(col("height_cm") == 0, None).otherwise(col("height_cm"))
).withColumn(
    "weight_kg",
    when(col("weight_kg") == 0, None).otherwise(col("weight_kg"))
).withColumn(
    "cholesterol_mg_dl",
    when(col("cholesterol_mg_dl") == 0, None).otherwise(col("cholesterol_mg_dl"))
)

In [ ]:
df.show()

+----------+--------+---+------+---------+---------+-----------+----------+----+-----------+------------+-----------------+
|student_id|    name|age|gender|height_cm|weight_kg|blood_group|department|year|systolic_bp|diastolic_bp|cholesterol_mg_dl|
+----------+--------+---+------+---------+---------+-----------+----------+----+-----------+------------+-----------------+
|      S001|  Minuri| 24|     F|     NULL|       84|         A-|        IT|   3|        114|          92|             NULL|
|      S002|  Methma| 21|     F|     NULL|       58|         B+|        IT|   4|        136|          86|              287|
|      S003|Thivanga| 22|     M|      188|     NULL|         O-|        IT|   2|        111|          81|              274|
|      S004|Maleesha| 22|     F|      171|     NULL|         O-|        IT|   2|        110|         102|              238|
|      S005| Randeep| 25|     M|     NULL|     NULL|         O+|        DS|   3|        150|          82|             NULL|
|      S

### Handle Missing Values (Option A – Drop Rows)

In [ ]:
df_clean = df.dropna()

In [ ]:
df_clean.show()

+----------+--------+---+------+---------+---------+-----------+----------+----+-----------+------------+-----------------+
|student_id|    name|age|gender|height_cm|weight_kg|blood_group|department|year|systolic_bp|diastolic_bp|cholesterol_mg_dl|
+----------+--------+---+------+---------+---------+-----------+----------+----+-----------+------------+-----------------+
|      S029|  Piyumi| 21|     F|      168|       74|        AB+|        DS|   4|        113|         102|              187|
|      S033| Shehani| 24|     F|      190|       95|        AB+|        CS|   3|        138|          76|              215|
|      S038| Hasitha| 22|     M|      180|       73|         O+|        CS|   3|        144|          69|              222|
|      S047|Tharindu| 21|     M|      167|       83|         O+|        DS|   2|        145|          75|              169|
|      S050|Maleesha| 23|     F|      180|       51|         O-|        DS|   2|        144|          89|              175|
|      S

# Part C – Feature Engineering

### Calculate BMI

In [ ]:
df_clean = df_clean.withColumn(
"BMI",
col("weight_kg") / ((col("height_cm")/100) ** 2)
)

### BMI Category

In [ ]:
df_clean = df_clean.withColumn(
"BMI_Category",
when(col("BMI") < 18.5, "Underweight")
.when(col("BMI") < 25, "Normal")
.when(col("BMI") < 30, "Overweight")
.otherwise("Obese")
)

### High Blood Pressure Flag

In [ ]:
from pyspark.sql.functions import col

df_clean = df_clean.withColumn(
    "high_bp",
    (col("systolic_bp") >= 140) | (col("diastolic_bp") >= 90)
)

### At-Risk Flag

In [ ]:
df_clean = df_clean.withColumn(
"at_risk",
(col("BMI_Category").isin("Overweight", "Obese")) |
(col("high_bp") == True) |
(col("cholesterol_mg_dl") >= 200)
)

# Part D – Analysis & Aggregation

### Average BMI per Department

In [ ]:
df_clean.groupBy("department").agg(avg("BMI").alias("avg_BMI")).show()

+----------+------------------+
|department|           avg_BMI|
+----------+------------------+
|        IT| 17.78197125852809|
|        CS|24.119883394401224|
|        DS|24.742234395181658|
+----------+------------------+



### Percentage of At-Risk Students per Department

In [ ]:
df_clean.groupBy("department") \
    .agg(
        ((sum(col("at_risk").cast("int")) / count("*")) * 100)
        .alias("at_risk_percentage")
    ) \
    .show()

+----------+------------------+
|department|at_risk_percentage|
+----------+------------------+
|        IT|             100.0|
|        CS| 85.71428571428571|
|        DS|             100.0|
+----------+------------------+



### Mean BP by Year

In [ ]:
df_clean.groupBy("year") \
    .agg(
        avg("systolic_bp").alias("avg_systolic"),
        avg("diastolic_bp").alias("avg_diastolic")
    ) \
    .show()

+----+------------------+-------------+
|year|      avg_systolic|avg_diastolic|
+----+------------------+-------------+
|   3|142.33333333333334|         78.0|
|   4|             128.6|         87.6|
|   2|141.33333333333334|         82.0|
+----+------------------+-------------+



### Pivot Table: Department × Gender → Mean BMI

In [ ]:
df_clean.groupBy("department") \
    .pivot("gender") \
    .agg(avg("BMI")) \
    .na.fill(0) \
    .show()

+----------+------------------+------------------+
|department|                 F|                 M|
+----------+------------------+------------------+
|        IT|               0.0| 17.78197125852809|
|        CS| 24.94235234483436|23.023258127157035|
|        DS|22.521950953307478| 29.18280127893002|
+----------+------------------+------------------+



### Sorted At-Risk Students List

In [ ]:
at_risk_df = df_clean.filter(col("at_risk") == True) \
    .select(
        "student_id",
        "name",
        "department",
        "BMI",
        "BMI_Category",
        "systolic_bp",
        "diastolic_bp",
        "cholesterol_mg_dl"
    ) \
    .orderBy(desc("BMI"))

at_risk_df.show()

+----------+--------+----------+------------------+------------+-----------+------------+-----------------+
|student_id|    name|department|               BMI|BMI_Category|systolic_bp|diastolic_bp|cholesterol_mg_dl|
+----------+--------+----------+------------------+------------+-----------+------------+-----------------+
|      S059|Maleesha|        CS|33.310844999156686|       Obese|        119|          72|              180|
|      S047|Tharindu|        DS|29.760837606224676|  Overweight|        145|          75|              169|
|      S100|Chathura|        DS|28.604764951635367|  Overweight|        130|          82|              164|
|      S061| Shehani|        DS|26.897670187097027|  Overweight|        156|          68|              274|
|      S033| Shehani|        CS|26.315789473684212|  Overweight|        138|          76|              215|
|      S029|  Piyumi|        DS|26.218820861678008|  Overweight|        113|         102|              187|
|      S083|   Nipun|       

# Part E – Exporting Results

In [ ]:
output_path = "/content/drive/MyDrive/BigData/at_risk_students.csv"
at_risk_df.coalesce(1).write.csv(output_path, header=True, mode="overwrite")